In [1]:
from __future__ import annotations
import os
os.environ['HF_HOME'] = "~/.cache"

import argparse
from pathlib import Path

import cv2
import torch
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, StableDiffusionXLImg2ImgPipeline
from peft import PeftModel
from PIL import Image


/home/alex/study/pcb-generation/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONTROLNET_PATH = "/home/alex/study/pcb-generation/trained/controlnet_600x600"
LORA_PARH = "/home/alex/study/pcb-generation/trained/lora_600x600"
PRETRAINED_MODEL_PATH = "stabilityai/stable-diffusion-xl-base-1.0"
LORA_SCALE = 0.85

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weight_dtype = torch.float16 if device.type == "cuda" else torch.float32

controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_PATH,
    torch_dtype=weight_dtype,
)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    PRETRAINED_MODEL_PATH,
    controlnet=controlnet,
    torch_dtype=weight_dtype,
    low_cpu_mem_usage=True
)

pipe.unet = PeftModel.from_pretrained(pipe.unet, LORA_PARH)
for module in pipe.unet.modules():
    if hasattr(module, "scaling"):
        for key in module.scaling:
            module.scaling[key] = LORA_SCALE
pipe.unet.eval()

pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()


Loading pipeline components...: 100%|██████████| 7/7 [02:54<00:00, 24.93s/it]
/home/alex/study/pcb-generation/venv/lib/python3.11/site-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
/home/alex/study/pcb-generation/venv/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2263: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLControlNetPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [4]:

STRUCTURE_MAP_PATH = "/home/alex/study/pcb-generation/images/structure_maps/layout_6.png"
OUTPUT_PATH = "/home/alex/study/pcb-generation/images/generated/img6.png"
PROMPT = "macro photo of printed circuit board, green solder mask, copper traces, realistic industrial PCB"
NEGATIVE_PROMPT = "blurry, noise, grain, jpeg artifacts, low quality, watermark, text, distorted, out of focus, overexposed, pixelated"

NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 8.0
CONTROLNET_CONDITIONING_SCALE = 0.8
HEIGHT = 600
WIDTH = 600
SEED = 42

structure = cv2.imread(STRUCTURE_MAP_PATH, cv2.IMREAD_GRAYSCALE)
if structure is None:
    raise FileNotFoundError(f"Failed to read structure map: {STRUCTURE_MAP_PATH}")
structure = cv2.resize(structure, (WIDTH, HEIGHT))
structure_image = Image.fromarray(structure)

generator = torch.Generator(device=device.type).manual_seed(SEED)
result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT or None,
    image=structure_image,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    controlnet_conditioning_scale=CONTROLNET_CONDITIONING_SCALE,
    height=HEIGHT,
    width=WIDTH,
    generator=generator,
).images[0]

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
result.save(output_path)
print(f"Saved generated image to: {output_path}")

100%|██████████| 50/50 [19:49<00:00, 23.79s/it]
/home/alex/study/pcb-generation/venv/lib/python3.11/site-packages/diffusers/pipelines/controlnet/pipeline_controlnet_sd_xl.py:928: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


Saved generated image to: /home/alex/study/pcb-generation/images/generated/img6.png


In [ ]:
# device = "cuda"

# refine_pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
#     "stabilityai/stable-diffusion-xl-base-1.0",
#     torch_dtype=torch.float16
# ).to(device)

# # pipe.enable_xformers_memory_efficient_attention()

# # загружаем LoRA
# lora_dir = "/mnt/ssdm2/users/alexblokh/pcb_generation/pcb-generation/pcb_lora"
# lora_scale = 1.0

# pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_dir)
# for module in pipe.unet.modules():
#     if hasattr(module, "scaling"):
#         for key in module.scaling:
#             module.scaling[key] = lora_scale
# pipe.unet.eval()

# image = Image.open('/mnt/ssdm2/users/alexblokh/pcb_generation/pcb-generation/images/layouts/layout_0.png')

# result = pipe(
#     prompt="""
# ultra detailed macro photo of a real printed circuit board,
# green solder mask, copper traces, vias, electronic components,
# realistic shadows, reflections, photorealistic, high detail, 8k
# """,
#     image=image,
#     strength=0.5,              # КЛЮЧЕВОЙ параметр
#     guidance_scale=7,
#     num_inference_steps=50
# ).images[0]

# result.save("/mnt/ssdm2/users/alexblokh/pcb_generation/pcb-generation/images/generated_img2img/pcb.png")